# Synthetic NER Data Generation using GLiNER

This notebook generates synthetic NER annotations using the multilingual GLiNER model.

We use:
- raw text reconstructed from factRuEval‑2016,
- GLiNER entity predictions (PER / ORG / LOC),
- conversion of character-level spans into BIO labels aligned with tokens.

The resulting synthetic dataset will be merged with the original factRuEval‑2016 data and used for additional ModernBERT fine‑tuning.

In [ ]:
import torch
from datasets import Dataset, DatasetDict

from src.synthetic.gliner_synthetic import (
    load_gliner,
    gliner_char_spans,
    generate_synthetic_labels
)

## Reconstruct raw text from tokens

We convert tokenized factRuEval samples into plain text for GLiNER inference.

In [ ]:
def reconstruct_text(example):
    text, offsets = tokens_to_text_and_offsets(example["tokens"])
    return {"text": text, "offsets": offsets}

text_ds = ds_small.map(reconstruct_text)

## Load GLiNER model

In [ ]:
gliner = load_gliner(device=DEVICE)

## Generate synthetic BIO labels using GLiNER

We:
1. run GLiNER to obtain character-level spans,
2. convert spans to BIO labels aligned with tokens,
3. create a synthetic dataset compatible with ModernBERT training.

In [ ]:
synthetic_ds = generate_synthetic_labels(
    text_ds,
    gliner_model=gliner,
    label_set=ENTITY_TYPES
)

synthetic_ds

## Merge synthetic and original datasets

We combine:
- original factRuEval‑2016 labels,
- synthetic GLiNER labels.

This creates an augmented dataset for additional fine‑tuning.

In [ ]:
augmented_ds = DatasetDict(
    {
        "train": ds_small["train"] + synthetic_ds["train"],
        "validation": ds_small["validation"],
        "test": ds_small["test"]
    }
)

augmented_ds

## Summary

We generated synthetic NER annotations using GLiNER and merged them with the original factRuEval‑2016 dataset.

This augmented dataset will be used in the next notebook for ModernBERT fine‑tuning with synthetic supervision.